## 1. 라이브러리 호출

In [2]:
import math
import time
from getpass import getpass                     # 인증키가 화면에 표시되지 않도록 하는 라이브러리
from urllib.parse import unquote                # URL을 보기 좋게 만들어주는 라이브러리

import requests
import pandas as pd

## 2. 인증키 입력

In [3]:
general_key = getpass("공공데이터포털 인증키를 입력하세요: ").strip()

SERVICE_KEY = unquote(general_key)

## 3. API 주소와 기본 요청변수 설정

In [4]:
API_URL = ("http://apis.data.go.kr/5080000/pricesInfoService/getPricesInfo")

# 한 번 호출할 때 가져오는 데이터 수
NUM_OF_ROWS = 100

## 4. 첫 페이지 요청하기
- 요청 메세지 명세서 활용

In [7]:
params = {
    "serviceKey" : SERVICE_KEY,
    "pageNo" : 1,
    "numOfRows" : NUM_OF_ROWS
}

response = requests.get(API_URL, params = params, timeout = 30)

## 5. 응답 상태 확인하기

In [8]:
print("HTTP 상태코드: ", response.status_code)
print("응답 형식: ", response.headers.get("Content-Type"))

HTTP 상태코드:  200
응답 형식:  application/json


## 6. JSON응답을 파이썬 객체로 바꿔주기

In [9]:
data = response.json()

## 7. header와 body 구분하기
- JSON 에서는 header와 body를 구분해야 함

In [10]:
header = data.get("header", {})
body = data.get("body", [])

In [11]:
print("결과 코드:", header.get("resultCode"))
print("전체 데이터 수:", header.get("totalCount"))
print("현재 페이지:", header.get("pageNo"))
print("결과 메시지:", header.get("resultMsg"))
print("현재 수집 건수:", len(body))

결과 코드: 00
전체 데이터 수: 675
현재 페이지: 1
결과 메시지: NORMAL_SERVICE
현재 수집 건수: 100


## 8. 첫 페이지 데이터 일부 확인

In [13]:
body[0:4]

[{'prdlst_nm': 'pc방 이용료',
  'stndrd_unit': 'A4 10매',
  'se': '고아읍',
  'pc': 7000,
  'date_stdde': '2023-06-25'},
 {'prdlst_nm': 'pc방 이용료',
  'stndrd_unit': 'A4 10매',
  'se': '도량동+선주원남동',
  'pc': 8000,
  'date_stdde': '2023-06-25'},
 {'prdlst_nm': 'pc방 이용료',
  'stndrd_unit': 'A4 10매',
  'se': '상모사곡동+임오동',
  'pc': 7000,
  'date_stdde': '2023-06-25'},
 {'prdlst_nm': 'pc방 이용료',
  'stndrd_unit': 'A4 10매',
  'se': '선산읍',
  'pc': 7000,
  'date_stdde': '2023-06-25'}]

# 전체 데이터 수집하기

## 1. 전체 페이지 수 계산하기
- 안전한 코드 구성을 위해 `get` 함수에 `,0` 추가
- nullif 같은 느낌?

In [15]:
total_count = int(header.get("totalCount", 0))

total_pages = math.ceil(total_count / NUM_OF_ROWS)

total_pages

7

## 2. 첫 번째 데이터를 누적 리스트에 저장하기

`isinstance()`
- 자료형 확인하는 함수
- 문법 : isinstance(확인할 값, 확인할 자료형)

In [16]:
if isinstance(body, dict): 
    body = [body]

all_records = body.copy()

print(len(all_records))

100


## 3. 두번째 데이터부터 수집하기

In [26]:
for page_no in range (2, total_pages + 1):

    # 현재 가져올 페이지 번호를 요청 변수에 저장
    params = {
    "serviceKey" : SERVICE_KEY,
    "pageNo" : page_no,
    "numOfRows" : NUM_OF_ROWS
    }
    # 현재 페이지의 데이터를 API를 통해서 요청하기
    response = requests.get(API_URL, params = params, timeout = 30)

    # HTTP 요청에 문제가 있으면 오류 발생하고 중단
    response.raise_for_status()

    # JSON 데이터 타입을 PYTHON으로 변환
    page_data = response.json()

    # header 와 body 부분 분할해서 가져오기 -> 위의 7번과 유사한 것인가?
    page_header = page_data["header"]
    page_body = page_data["body"]

    # API 내부 결과가 정상이 아닐 때 종료하는 코드 -> 00을 숫자로 받아올 수 있을 지 모르니까 안전하게
    if str(page_header["resultCode"]) != "00":
        print(f"{page_no} 페이지 호출 실패: ", page_header["resultMsg"])
        break

    # 현재 페이지 데이터를 전체 데이터 리스트에 추가
    all_records.extend(page_body)

    # 서버에 너무 부담을 주지 않기 위해 휴식
    time.sleep(1)

### append VS extend
- append: 리스트.append(리스트2) 하면 리스트2가 리스트 형태로 들어가서 하나만 추가됨
- extend: 리스트 안에 있는 것들이 하나씩 벗겨지며 들어감 원래 리스트안에 리스트2의 갯수까지 다 추가되는 느낌 
a.append(b) = [1,2,[3,4]]
a.extend[b] = [1,2,3,4]


## 우리가 보기 편한 형태로 변환

In [27]:
df = pd.DataFrame(all_records)

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   prdlst_nm    1275 non-null   str  
 1   stndrd_unit  1275 non-null   str  
 2   se           1275 non-null   str  
 3   pc           1275 non-null   int64
 4   date_stdde   1275 non-null   str  
dtypes: int64(1), str(4)
memory usage: 49.9 KB


In [29]:
df.head(5)

,prdlst_nm,stndrd_unit,se,pc,date_stdde
0,pc방 이용료,A4 10매,고아읍,7000,2023-06-25
1,pc방 이용료,A4 10매,도량동+선주원남동,8000,2023-06-25
2,pc방 이용료,A4 10매,상모사곡동+임오동,7000,2023-06-25
3,pc방 이용료,A4 10매,선산읍,7000,2023-06-25
4,pc방 이용료,A4 10매,송정동+형곡동,7000,2023-06-25


## 컬럼명 변경

In [30]:
df.columns

Index(['prdlst_nm', 'stndrd_unit', 'se', 'pc', 'date_stdde'], dtype='str')

In [31]:
column_names = {
    'prdlst_nm' : "품목명", 
    'stndrd_unit' : "규격_단위", 
    'se' : "규격", 
    'pc' : "가격", 
    'date_stdde' : "기준일자"
}

df = df.rename(columns = column_names)

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1275 entries, 0 to 1274
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   품목명     1275 non-null   str  
 1   규격_단위   1275 non-null   str  
 2   규격      1275 non-null   str  
 3   가격      1275 non-null   int64
 4   기준일자    1275 non-null   str  
dtypes: int64(1), str(4)
memory usage: 49.9 KB


### 데이터 csv 파일로 저장하기

In [33]:
file_name = "구미시_가격_정보.csv"

df.to_csv(file_name, index = False, encoding = 'utf=8-sig')